# RAM Finetune on Kaggle (Flickr30k)
**Flow:** Install deps → Clone repo → Link data & weights → Mini test → Full finetune → Evaluate

**Trước khi chạy:** Add 2 input sources trong Kaggle Settings → Add Data:
1. **Dataset:** `hsankesara/flickr-image-dataset`
2. **Model:** `nguyentrthinh/ram-pretrained`

## 1. Install dependencies

In [ ]:
!pip install -q timm==0.4.12 "transformers>=4.25.1,<=4.55.4" fairscale==0.4.4 \
    pycocoevalcap ruamel.yaml scipy Pillow
!pip install -q git+https://github.com/openai/CLIP.git

## 2. Clone repo

In [ ]:
# ===== CONFIG =====
GITHUB_REPO = "https://github.com/YOUR_USERNAME/image_tagging.git"  # <-- sửa URL repo
# ==================

In [ ]:
import os
os.chdir("/kaggle/working")

if not os.path.isdir("image_tagging"):
    !git clone {GITHUB_REPO}
else:
    print("Repo already cloned")

os.chdir("image_tagging/recognize-anything")
!pwd

## 3. Link pretrained weights & images từ Kaggle Input
Không cần download — dùng trực tiếp từ Kaggle Input.

In [ ]:
# ===== KAGGLE INPUT PATHS =====
# Pretrained RAM model (from Kaggle Model: nguyentrthinh/ram-pretrained)
KAGGLE_PRETRAINED = "/kaggle/input/ram-pretrained/pytorch/default/1/ram_swin_large_14m.pth"

# Flickr30k images (from Kaggle Dataset: hsankesara/flickr-image-dataset)
KAGGLE_IMAGES_DIR = "/kaggle/input/flickr-image-dataset/flickr30k_images/flickr30k_images"
# ==============================

In [ ]:
# Symlink pretrained weights
os.makedirs("pretrained", exist_ok=True)
pretrained_link = "pretrained/ram_swin_large_14m.pth"
if not os.path.exists(pretrained_link):
    os.symlink(KAGGLE_PRETRAINED, pretrained_link)
    print(f"Linked: {pretrained_link} -> {KAGGLE_PRETRAINED}")
else:
    print(f"Already exists: {pretrained_link}")

# Verify
!ls -lh pretrained/

In [ ]:
# Symlink Flickr30k images
os.makedirs("batch_image/archive/flickr30k_images", exist_ok=True)
images_link = "batch_image/archive/flickr30k_images/flickr30k_images"
if not os.path.exists(images_link):
    os.symlink(KAGGLE_IMAGES_DIR, images_link)
    print(f"Linked: {images_link} -> {KAGGLE_IMAGES_DIR}")
else:
    print(f"Already exists: {images_link}")

# Verify
imgs = os.listdir(images_link)
print(f"Images found: {len(imgs)}")
print("Sample:", imgs[:5])

In [ ]:
# Verify train/val JSON đã có trong repo
import json

for split in ["train", "val", "train_mini"]:
    path = f"datasets/flickr30k/{split}.json"
    if os.path.isfile(path):
        data = json.load(open(path, "r", encoding="utf-8"))
        print(f"{split}: {len(data)} samples")
    else:
        print(f"{split}: NOT FOUND")

## 4. GPU check + auto config

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU not available! Enable GPU: Settings -> Accelerator -> GPU T4 x2"

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
n_gpus = torch.cuda.device_count()

# Auto batch size theo VRAM
if gpu_mem_gb >= 40:    batch_size = 26
elif gpu_mem_gb >= 20:  batch_size = 12
elif gpu_mem_gb >= 15:  batch_size = 8
elif gpu_mem_gb >= 10:  batch_size = 4
else:                   batch_size = 2

print(f"GPU: {gpu_name} | VRAM: {gpu_mem_gb:.1f}GB | Count: {n_gpus} | Batch size: {batch_size}")

## 5. Mini test (10 samples, 1 epoch)
Verify code chạy trước khi train full.

In [ ]:
!mkdir -p output/test_mini

!CUDA_VISIBLE_DEVICES=0 python -m torch.distributed.launch \
    --nproc_per_node=1 \
    finetune.py \
    --config datasets/flickr30k/finetune_config_mini.yaml \
    --model-type ram \
    --checkpoint pretrained/ram_swin_large_14m.pth \
    --output-dir output/test_mini

print("\n" + "="*50)
print("MINI TEST DONE!")
print("="*50)
!cat output/test_mini/log.txt

## 6. Full finetune
Chỉ chạy sau khi mini test thành công.

In [ ]:
# Ghi config với batch_size đã auto-detect
config_yaml = f"""train_file:
  - 'datasets/flickr30k/train.json'

image_path_root: "batch_image/archive/flickr30k_images/flickr30k_images"

vit: 'swin_l'
vit_grad_ckpt: True
vit_ckpt_layer: 0

image_size: 384
batch_size: {batch_size}

weight_decay: 0.05
init_lr: 5e-06
min_lr: 0
max_epoch: 5
warmup_steps: 500

class_num: 4585
"""

with open("datasets/flickr30k/finetune_config_kaggle.yaml", "w") as f:
    f.write(config_yaml)

print(f"Config saved (batch_size={batch_size})")
print(config_yaml)

In [ ]:
!mkdir -p output/flickr30k_finetune

n_gpus = torch.cuda.device_count()

!CUDA_VISIBLE_DEVICES=0 python -m torch.distributed.launch \
    --nproc_per_node={n_gpus} \
    finetune.py \
    --config datasets/flickr30k/finetune_config_kaggle.yaml \
    --model-type ram \
    --checkpoint pretrained/ram_swin_large_14m.pth \
    --output-dir output/flickr30k_finetune

In [ ]:
# Training log
!cat output/flickr30k_finetune/log.txt
print()
!ls -lh output/flickr30k_finetune/*.pth

## 7. Resume training (nếu session bị ngắt)
Kaggle T4 chỉ có 12h. Nếu bị ngắt giữa chừng, chạy cell này.

In [ ]:
# Uncomment để resume
# !CUDA_VISIBLE_DEVICES=0 python -m torch.distributed.launch \
#     --nproc_per_node=1 \
#     finetune.py \
#     --config datasets/flickr30k/finetune_config_kaggle.yaml \
#     --model-type ram \
#     --checkpoint pretrained/ram_swin_large_14m.pth \
#     --resume output/flickr30k_finetune/checkpoint_latest.pth \
#     --output-dir output/flickr30k_finetune

## 8. Evaluate: Pretrained vs Finetuned

In [ ]:
!python evaluate.py \
    --checkpoint pretrained/ram_swin_large_14m.pth \
    --checkpoint-b output/flickr30k_finetune/checkpoint_best.pth \
    --val-json datasets/flickr30k/val.json \
    --image-root batch_image/archive/flickr30k_images/flickr30k_images \
    --model-type ram \
    --max-samples 500 \
    --output output/flickr30k_finetune/eval_results.json

## 9. Quick inference test

In [ ]:
import torch
from PIL import Image
from ram.models import ram
from ram import inference_ram, get_transform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_ckpt = "output/flickr30k_finetune/checkpoint_best.pth"
if os.path.isfile(best_ckpt):
    ckpt_data = torch.load(best_ckpt, map_location=device)
    model = ram(pretrained="pretrained/ram_swin_large_14m.pth", image_size=384, vit="swin_l")
    model.load_state_dict(ckpt_data["model"], strict=False)
    model = model.to(device).eval()
    transform = get_transform(image_size=384)

    # Test vài ảnh từ val set
    import json
    val_data = json.load(open("datasets/flickr30k/val.json", "r", encoding="utf-8"))[:5]
    for entry in val_data:
        img_path = os.path.join("batch_image/archive/flickr30k_images/flickr30k_images", entry["image_path"])
        if os.path.isfile(img_path):
            img = transform(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
            tags_en, tags_cn = inference_ram(img, model)
            print(f"{entry['image_path']}: {tags_en}")
else:
    print(f"Checkpoint not found: {best_ckpt}")

## 10. Download results

In [ ]:
!tar -czf /kaggle/working/finetune_result.tar.gz \
    output/flickr30k_finetune/checkpoint_best.pth \
    output/flickr30k_finetune/checkpoint_latest.pth \
    output/flickr30k_finetune/log.txt \
    output/flickr30k_finetune/eval_results.json

!ls -lh /kaggle/working/finetune_result.tar.gz
print("\nDownload: Kaggle sidebar -> Output -> finetune_result.tar.gz")